# 01 Spark 数据清洗

**目标**：用 Spark DataFrame API 处理同一份客户流失数据。

和 pandas 版本相比，写法几乎一样，区别在于两点：
1. 数据是**分布式**的，操作会被翻译成执行计划，延迟到 action 才真正计算
2. 有 **Web UI**（默认 4040 端口），能看到 DAG、Stage、Shuffle 读写量

第二点是「大数据」与「单机」最直观的区别——验收演示时务必打开看一眼。

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import re

spark = (SparkSession.builder
         .appName('churn-clean')
         .master('local[*]')
         .config('spark.sql.shuffle.partitions', '8')
         # Spark 4.0 起默认开启 ANSI 模式：cast 遇到 'N/A' 这类非法值会直接抛
         # CAST_INVALID_INPUT，而不是返回 null。关掉它，cast 才等价于 pandas 的
         # errors='coerce'。不想关也行，把下面的 cast 全换成 try_cast。
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark 版本:', spark.version)
print('ANSI 模式:', spark.conf.get('spark.sql.ansi.enabled'))

# 这里打印出来的是容器内主机名（如 f04bb449a9dd），在 Windows 宿主机上解析不了，
# 演示时必须换成 localhost。注意只有 4040 映射到了宿主机，落到 4041 就打不开了。
ui = spark.sparkContext.uiWebUrl or ''
print('Spark UI :', ui)
if ui:
    print('宿主机打开:', re.sub(r'//[^:/]+', '//localhost', ui))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 23:30:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 版本: 4.0.4
ANSI 模式: false
Spark UI : http://f04bb449a9dd:4040


## 1. 读取原始数据

注意 CSV 带 BOM，第一列名会多一个不可见字符，需要清掉。

In [2]:
df = (spark.read
      .option('header', True)
      .option('inferSchema', True)
      .option('encoding', 'UTF-8')
      .csv('../../data/raw/customer_churn.csv'))

df = df.toDF(*[c.replace('\ufeff', '').strip() for c in df.columns])
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charge: double (nullable = true)
 |-- total_charge: string (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- support_calls: integer (nullable = true)
 |-- is_active: integer (nullable = true)
 |-- churn: integer (nullable = true)



In [3]:
print('原始行数:', df.count())
df.select('customer_id', 'age', 'monthly_charge', 'total_charge', 'churn').show(5)

原始行数: 1545
+-----------+---+--------------+------------+-----+
|customer_id|age|monthly_charge|total_charge|churn|
+-----------+---+--------------+------------+-----+
|     C01225| 22|         35.39|     1521.77|    0|
|     C01388| 53|        140.14|     6026.02|    1|
|     C00940| 31|        104.32|     6467.84|    0|
|     C00624| 56|         40.84|     2899.64|    0|
|     C01265| 48|        103.23|     1961.37|    0|
+-----------+---+--------------+------------+-----+
only showing top 5 rows


## 2. 清洗

Spark 里 `cast` 遇到无法转换的值会返回 null，效果等价于 pandas 的 `errors='coerce'`。
所以类型转换本身就是一步缺失值清洗。

> **Spark 4.0 的坑（实测）**：4.0 起默认开启 ANSI 模式，`spark.sql.ansi.enabled = true`。
> 这时 `cast` 碰到 `N/A` 不会返回 null，而是直接抛
> `[CAST_INVALID_INPUT] The value 'N/A' of the type "STRING" cannot be cast to "DOUBLE"`。
> 这一列是 `total_charge`——`inferSchema` 看到 `N/A` 就把它推断成 string，其余数值列不受影响。
>
> 两种修法二选一：
>
> - 建会话时 `.config('spark.sql.ansi.enabled', 'false')`（本 notebook 用的这种）
> - 或者保持 ANSI，把 `cast` 换成 `try_cast`：`F.expr('try_cast(total_charge as double)')`


In [4]:
df2 = (df
       .withColumn('age', F.col('age').cast(DoubleType()))
       .withColumn('monthly_charge', F.col('monthly_charge').cast(DoubleType()))
       .withColumn('total_charge', F.col('total_charge').cast(DoubleType()))
       .withColumn('tenure_months', F.col('tenure_months').cast(DoubleType()))
       .withColumn('support_calls', F.col('support_calls').cast(DoubleType()))
       .withColumn('is_active', F.col('is_active').cast(DoubleType()))
       .withColumn('churn', F.col('churn').cast(IntegerType())))

df2.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in ['age', 'monthly_charge', 'total_charge']]).show()

+---+--------------+------------+
|age|monthly_charge|total_charge|
+---+--------------+------------+
| 75|            70|          70|
+---+--------------+------------+



## 3. 异常值过滤与去重

Spark 是惰性求值，这里只是构建执行计划，直到 `count()` 才真正跑。

> **和 pandas 版行数不同是有意的**：pandas 版把异常值**置为缺失**、保留样本，
> 所以去重后是 1500 行；这里直接**剔除**异常行，结果是 **1440 行**。
> 两种策略业务上都成立——前者不丢样本，后者不留脏值。被问到时这正好是一个可展开的点。


In [5]:
df3 = (df2
       .filter((F.col('age').isNull()) | (F.col('age') <= 100))
       .filter((F.col('monthly_charge').isNull()) | (F.col('monthly_charge') >= 0))
       .filter((F.col('churn').isNotNull()))
       .dropDuplicates(['customer_id']))

print('清洗后行数:', df3.count())
print('流失比例:', df3.select(F.avg('churn')).first()[0])

清洗后行数: 1440
流失比例: 0.17222222222222222


## 4. 衍生特征

In [6]:
df4 = (df3
       .withColumn('avg_monthly',
                   # total_charge 缺失时退回当月月费，别让它除出 null
                   # （pandas 版 fillna(monthly_charge) 是同一个兜底）
                   F.when((F.col('tenure_months') > 0) & F.col('total_charge').isNotNull(),
                          F.col('total_charge') / F.col('tenure_months'))
                    .otherwise(F.col('monthly_charge')))
       .withColumn('support_per_year',
                   F.col('support_calls') /
                   F.greatest(F.col('tenure_months') / 12, F.lit(1))))

print('avg_monthly 空值:', df4.filter(F.col('avg_monthly').isNull()).count(),
      '（这几行月费和累计费用都缺，兜不住，交给下游 fillna）')
df4.select('customer_id', 'tenure_months', 'total_charge',
           'avg_monthly', 'support_per_year').show(5)


avg_monthly 空值: 5 （这几行月费和累计费用都缺，兜不住，交给下游 fillna）
+-----------+-------------+------------+------------------+------------------+
|customer_id|tenure_months|total_charge|       avg_monthly|  support_per_year|
+-----------+-------------+------------+------------------+------------------+
|     C00940|         62.0|     6467.84|104.32000000000001|1.9354838709677418|
|     C00394|         11.0|      1052.7|              95.7|               4.0|
|     C00460|         22.0|     2088.24| 94.91999999999999| 1.090909090909091|
|     C00566|         34.0|     1660.22|             48.83|0.3529411764705882|
|     C00373|         34.0|     1418.14|             41.71|0.7058823529411764|
+-----------+-------------+------------+------------------+------------------+
only showing top 5 rows


## 5. 写出 Parquet

> Windows 下 PySpark 写文件需要 Hadoop 的 `winutils.exe`，容器（Linux）里不需要。
> 如果在 Windows 本地跑报 `HADOOP_HOME` 相关错误，直接在容器里跑这一格。

In [7]:
import os
out = '../../data/processed/spark_clean.parquet'
os.makedirs('../../data/processed', exist_ok=True)

try:
    (df4.repartition(4)
        .write.mode('overwrite')
        .parquet(out))
    print('已写入:', out)
except Exception as e:
    print('写入失败（Windows 常见，缺 winutils）:', type(e).__name__)
    print('容器内运行即可正常写出')

已写入: ../../data/processed/spark_clean.parquet


## 小结

- Spark 的 API 和 pandas 很像，但**执行是惰性的**：`filter`/`withColumn` 只建计划，`count`/`show`/`write` 才触发计算
- 打开 `http://localhost:4040` 能看到 DAG 图、Stage 划分、Shuffle 读写量——这是大数据作业的「体检报告」
- 下一步：用 Spark MLlib 训练，走完整的 Pipeline